# colony4b

> Phase 18, Stage 4b (`specs/multiagent/phase18/stage4.md`, a user-approved deviation, 2026-10-25): the three Stage 4 designs (guard copies, failure flags, squads of 8) at 32 agents with independent liars, so each design's trend rests on 16, 32 and 64 agents. The plain board and guards at 32 agents are the 32-agent run's cells, replayed from the cache. `stage4bGrid` (24 cells), `stage4bPilotGrid`, `reusedCells4b`, `pilotCheck4b`, `STAGE4B_PLAN`, `stage4bSummaries` and `writeStage4b`.

In [ ]:
#| default_exp colony4b

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np

from explRL.bench_stats import holm
from explRL.colony4 import ARMS4, _cell, stage4Estimate
from explRL.colony4_report import CELL_KEYS, _test, _two, diffs
from explRL.colony32 import stage32Grid
from explRL.colony_grid import COLONY_SEEDS, PILOT_SEED
from explRL.colony_report import NAN, trendSlopes

N4B = 32
ARMS4B = ('guardDeal', 'failFlag', 'squads')
HYP4B = (('H18_16', 'guardDeal'), ('H18_17', 'failFlag'), ('H18_18', 'squads'))
TRENDS4B = (('colonyGuards', (8, 16, 32, 64)),) + tuple((a, (16, 32, 64)) for a in ARMS4B)


def stage4bGrid(seeds=COLONY_SEEDS):
    "Study colony4b: the three designs at 32 agents with independent liars, Stage 1's seeds (24 cells; 23,040 calls)."
    return [_cell('colony4b', a, N4B, s, 'independent') for a in ARMS4B for s in seeds]


def stage4bPilotGrid():
    "The pilot (not analysed): the three designs at 32 agents, seed 1890."
    return [_cell('colony4bPilot', a, N4B, PILOT_SEED, 'independent') for a in ARMS4B]


def reusedCells4b(seeds=COLONY_SEEDS):
    "The 16 references: the 32-agent run's plain board and guards with independent liars (study colony32), replayed from the cache."
    return [c for c in stage32Grid(seeds) if c['level'] in ('plain|independent', 'colonyGuards|independent')]


STAGE4B_PLAN = dict(
    version=1, frozen=True, plan='specs/multiagent/phase18/stage4.md (Stage 4b)',
    deviation='user-approved 2026-10-25: each Stage 4 design ran at 16 and 64 agents only; 32 agents shows the shape of each trend',
    arms={a: ARMS4[a] for a in ARMS4B}, cells={N4B: [f'{a}|independent' for a in ARMS4B]}, seeds=COLONY_SEEDS,
    reused={N4B: ['plain|independent', 'colonyGuards|independent']},
    primary=dict(H18_16='guardDeal', H18_17='failFlag', H18_18='squads',
                 test='arm - colonyGuards, honestPooled, independent liars, N 32, 8 seeds paired: one-sided exact sign-flip > 0; '
                      'Holm over the three', alpha=0.05),
    robustness='each primary repeated on welfareHonest; an estimate of the opposite sign marks the primary not robust',
    descriptive=dict(trend='per design, each seed\'s least-squares slope over log2 N of (design - plain board), guards at 8-64 and the '
                           'designs at 16, 32 and 64; mean and two-sided exact sign-flip p; descriptive, since the 16 and 64 agent '
                           'results were seen before this plan', advantage='each design minus the plain board at 32 agents'),
    pilot=dict(cells=[f'{a}|independent N {N4B}' for a in ARMS4B], seed=PILOT_SEED, maxInvalid=0.05),
    costRule=dict(run=20.0, otherwise='stop and ask', projection="Stage 4b's calls x the pilot's cost per call by board kind"),
    forecast=dict(file='specs/multiagent/phase18/colony4b_predictions.json',
                  source='Stage 4\'s numpy forecast rows at 32 agents (results/multiagent/colony/forecast4, frozen at 1e9c53d)',
                  sha256='629f5c788a515401c47efb4c237b4fed085a5196d1df9342b5b4eb20a2859084'), cap=250.0, nboot=2000, exploratory=True)

In [ ]:
#| export
def pilotCheck4b(rows, plan=STAGE4B_PLAN):
    "The pilot's gate (invalid replies) and the cost rule -> dict(ok, invalid, perCall, projected, verdict)."
    f = lambda r, k: float(r[k])
    calls = sum(f(r, 'calls') for r in rows) or 1.0
    invalid = sum(f(r, 'invalid') for r in rows) / calls
    kind = lambda r: 'squads' if r['level'].startswith('squads') else f'N{N4B}'
    perCall = {k: sum(f(r, 'cost') for r in rows if kind(r) == k) / (sum(f(r, 'calls') for r in rows if kind(r) == k) or 1.0)
               for k in (f'N{N4B}', 'squads')}
    projected = stage4Estimate(stage4bGrid(), perCall)['cost']
    verdict = 'run' if projected <= plan['costRule']['run'] else 'stop'
    return dict(ok=invalid < plan['pilot']['maxInvalid'] and verdict == 'run', invalid=invalid, perCall=perCall,
                projected=float(projected), verdict=verdict)


def stage4bSummaries(rows4b, reused, prior=(), plan=STAGE4B_PLAN, nboot=2000):
    """H18-16 to H18-18 (each design minus guards at 32 agents, one-sided, Holm over the three) with their robustness on raw
    honest welfare; each design's advantage over the plain board at 32; each design's trend over team size (descriptive), with
    `prior` holding the rows at the other sizes (Stage 1, the 32- and 64-agent runs, Stage 4 and its references)."""
    rows, ind = list(rows4b) + list(reused), 'independent'
    hyp = {}
    for h, arm in HYP4B:
        t = _test(diffs(rows, arm, 'colonyGuards', 'honestPooled', ind, N4B), nboot)
        rob = _test(diffs(rows, arm, 'colonyGuards', 'welfareHonest', ind, N4B), nboot)
        hyp[h] = dict(t, arm=arm, robustness=rob, robust=bool(np.sign(rob['estimate']) == np.sign(t['estimate'])))
    for h, q in zip([h for h, _ in HYP4B], holm([hyp[h]['p'] for h, _ in HYP4B])):
        hyp[h].update(pHolm=float(q), supported=bool(q < plan['primary']['alpha'] and hyp[h]['estimate'] > 0))
    adv = {a: float(np.mean(list(d.values()))) for a in ('colonyGuards',) + ARMS4B
           if (d := diffs(rows, a, 'plain', 'honestPooled', ind, N4B))}
    trend = {}
    for a, sizes in TRENDS4B:
        sl = list(trendSlopes(list(prior) + rows, a, list(sizes)).values())
        trend[a] = dict(sizes=list(sizes), n=len(sl), slope=float(np.mean(sl)) if sl else NAN, p=_two(sl), rising=sum(v > 0 for v in sl))
    mean = lambda rs, k: float(np.nanmean([float(r[k]) for r in rs if r.get(k) not in ('', None)] or [NAN]))
    lv = sorted({r['level'] for r in rows})
    cells = {f'{l} N {N4B}': {k: mean([r for r in rows if r['level'] == l], k) for k in CELL_KEYS} for l in lv}
    return dict(hyp=hyp, advantage=adv, trend=trend, cells=cells)


_f = lambda x: 'n/a' if x is None or x != x else f'{x:.3f}'


def writeStage4b(path, S, meta):
    "report.md for Stage 4b."
    L = ['# Phase 18, Stage 4b: the Stage 4 designs at 32 agents (exploratory; a user-approved deviation)', '',
         f"plan {meta.get('planHash')}, commit {meta.get('commit')}, mode {meta.get('mode')}", '',
         '| test | n | design - colonyGuards (N 32) | 95% interval | p | Holm | supported | on raw welfare | robust |',
         '|---|---|---|---|---|---|---|---|---|']
    for h, t in S['hyp'].items():
        L.append(f"| {h} {t['arm']} | {t['n']} | {_f(t['estimate'])} | [{_f(t['ci'][0])}, {_f(t['ci'][1])}] | {_f(t['p'])} | "
                 f"{_f(t['pHolm'])} | {t['supported']} | {_f(t['robustness']['estimate'])} | {t['robust']} |")
    L += ['', 'Advantage over the plain board at 32 agents: ' + '; '.join(f'{a} {_f(v)}' for a, v in S['advantage'].items()), '',
          '| trend (descriptive) | sizes | n | slope per doubling | p (two-sided) | seeds rising |', '|---|---|---|---|---|---|']
    L += [f"| {a} | {'-'.join(map(str, t['sizes']))} | {t['n']} | {_f(t['slope'])} | {_f(t['p'])} | {t['rising']} |"
          for a, t in S['trend'].items()]
    keys = list(next(iter(S['cells'].values()))) if S['cells'] else []
    L += ['', '| cell | ' + ' | '.join(keys) + ' |', '|---|' + '---|' * len(keys)]
    L += [f'| {c} | ' + ' | '.join(_f(v) for v in d.values()) + ' |' for c, d in S['cells'].items()]
    open(path, 'w').write('\n'.join(L) + '\n')

## Tests

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
from explRL.colony4b import *
from explRL.colony4 import stage4Estimate
from explRL.fix_grid import cellCalls

# the grid: 3 designs x 8 seeds at 32 agents, a quarter of the agents liars; the calls are the Claude agents' turns
g = stage4bGrid()
assert len(g) == 24 and {c['N'] for c in g} == {32} and all(c['cfg']['nLiar'] == 8 for c in g)
assert {c['level'] for c in g} == {'guardDeal|independent', 'failFlag|independent', 'squads|independent'}
assert sum(cellCalls(c) for c in g) == 23040 and len(stage4bPilotGrid()) == 3
assert len(reusedCells4b()) == 16 and {c['study'] for c in reusedCells4b()} == {'colony32'}
assert not STAGE4B_PLAN['frozen'] or STAGE4B_PLAN['forecast']['sha256']

# the pilot's gate and cost rule
pr = [dict(level=f'{a}|independent', calls=960, invalid=5, cost=0.29 if a == 'squads' else 0.58) for a in ('guardDeal', 'failFlag', 'squads')]
v = pilotCheck4b(pr)
assert v['ok'] and abs(v['projected'] - (16 * 960 * 0.58 / 960 + 8 * 960 * 0.29 / 960)) < 1e-9
assert not pilotCheck4b([dict(r, invalid=100) for r in pr])['ok']

# the analysis on synthetic rows: every design beats guards at 32 in every seed; squads rise 0.04 per doubling from 16 to 64
def row(arm, N, s, v):
    return dict(level=f'{arm}|independent', N=N, seed=s, honestPooled=v, welfareHonest=v)
seeds = range(1800, 1808)
base = {'plain': 0.0, 'colonyGuards': 0.2, 'guardDeal': 0.25, 'failFlag': 0.26, 'squads': 0.3}
rows = [row(a, 32, s, v + 0.001 * s) for a, v in base.items() if a not in ('plain', 'colonyGuards') for s in seeds]
reused = [row(a, 32, s, base[a] + 0.001 * s) for a in ('plain', 'colonyGuards') for s in seeds]
prior = [row(a, N, s, base[a] + (0.04 * (np.log2(N) - 5) if a == 'squads' else 0) + 0.001 * s)
         for a in base for N in (8, 16, 64) for s in seeds if a in ('plain', 'colonyGuards') or N != 8]
S = stage4bSummaries(rows, reused, prior, nboot=200)
assert all(t['supported'] and t['robust'] and abs(t['p'] - 1 / 256) < 1e-12 for t in S['hyp'].values())
assert abs(S['hyp']['H18_18']['estimate'] - 0.1) < 1e-9 and abs(S['advantage']['squads'] - 0.3) < 1e-9
assert abs(S['trend']['squads']['slope'] - 0.04) < 1e-9 and S['trend']['squads']['rising'] == 8
assert S['trend']['colonyGuards']['sizes'] == [8, 16, 32, 64] and abs(S['trend']['colonyGuards']['slope']) < 1e-9
with tempfile.TemporaryDirectory() as tmp:
    writeStage4b(Path(tmp) / 'r.md', S, dict(planHash='x', commit='y', mode='test'))
    assert 'H18_18 squads' in (Path(tmp) / 'r.md').read_text()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()